In [58]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

In [36]:
features = pd.read_parquet('data/features.parquet')


In [37]:
item_ids = pd.concat([features['id1'], features['id2']], ignore_index=True)
item_counts = item_ids.value_counts()

In [38]:
repeated_ids = item_counts[item_counts > 1].index

In [39]:
repeat_mask = features['id1'].isin(repeated_ids) | features['id2'].isin(repeated_ids)
repeat_mask.sum()

np.int64(34775)

In [40]:
train_repeated = features[repeat_mask]
remaining = features[~repeat_mask]

train_remaining, test = train_test_split(remaining, test_size=0.2, random_state=42, stratify=remaining['target'])

train = pd.concat([train_repeated, train_remaining], ignore_index=True)

In [41]:
item_ids = pd.concat([test['id1'], test['id2']], ignore_index=True)
item_counts = item_ids.value_counts()
print((item_counts > 1).sum())

0


In [42]:
train_ids = set(train['id1']) | set(train['id2'])
test_ids = set(test['id1']) | set(test['id2'])

len(train_ids & test_ids)

0

In [43]:
train['target'].mean(), test['target'].mean()

(np.float64(0.254900193002491), np.float64(0.26524721953578334))

In [46]:
X_train, X_test = train.drop(columns=['target', 'id1', 'id2']), test.drop(columns=['target', 'id1', 'id2'])
y_train, y_test = train['target'], test['target']

In [49]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(299478, 11) (66176, 11) (299478,) (66176,)


In [55]:
model = CatBoostClassifier(
    iterations=300,
    depth=5,
    learning_rate=0.05,
    loss_function='Logloss',
    random_seed=42,
    verbose=50
)

model.fit(X_train, y_train)


0:	learn: 0.6761550	total: 333ms	remaining: 1m 39s
50:	learn: 0.5140596	total: 9.26s	remaining: 45.2s
100:	learn: 0.5052322	total: 14.3s	remaining: 28.2s
150:	learn: 0.5013728	total: 17s	remaining: 16.8s
200:	learn: 0.4987046	total: 21.7s	remaining: 10.7s
250:	learn: 0.4967982	total: 24.1s	remaining: 4.71s
299:	learn: 0.4952746	total: 27.8s	remaining: 0us


CatBoostClassifier(depth=5, iterations=300, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=50)

In [57]:
y_proba = model.predict_proba(X_test)[:, 1]


[0.58008269 0.32102019 0.16158848 ... 0.15000114 0.16445359 0.21831237]


In [59]:
print('ROC-AUC:', roc_auc_score(y_test, y_proba))
print('Average Precision:', average_precision_score(y_test, y_proba))
print('Logloss:', log_loss(y_test, y_proba))

ROC-AUC: 0.7348048670632481
Average Precision: 0.532577034094651
Logloss: 0.5032419405960976
